# Đánh giá Hệ thống Acne Advisor AI bằng RAGAS

Notebook chạy đánh giá định lượng toàn diện hệ thống **Acne Advisor AI** trên **50 câu hỏi chuẩn** theo phương pháp **RAGAS** (*Retrieval Augmented Generation Assessment*, Es et al., 2023, `arXiv:2309.15217`).

### Cấu hình chấm độc lập (tránh thiên vị tự chấm):
- **System LLM:** `gemini-3.5-flash-lite`
- **Judge LLM (NVIDIA NIM):** `nvidia/nemotron-3-ultra-550b-a55b` (550B / 55B active), dự phòng `mistralai/mistral-large-4-1t-a52b` (1T / 52B active)
- **Embedding Model:** `models/gemini-embedding-2` (3072 chiều)

### Các chỉ số được đo:
1. **Faithfulness** (*Độ trung thực với bằng chứng*): phát hiện ảo giác (hallucination).
2. **Answer Relevancy** (*Độ liên quan câu trả lời*): đo mức độ bám sát câu hỏi người dùng.
3. **Context Precision** (*Độ chính xác ngữ cảnh*): đo độ chính xác xếp hạng top ngữ cảnh truy hồi.
4. **Context Recall** (*Độ bao phủ ngữ cảnh*): đo ngữ cảnh truy hồi có đủ ý theo đáp án chuẩn.
5. **Answer Correctness** (*Độ chính xác câu trả lời*): kết hợp độ chính xác sự kiện (75%) và độ tương đồng ngữ nghĩa (25%) so với đáp án chuẩn.
6. **Safe Abstention Accuracy** (*Tỷ lệ từ chối an toàn*): kiểm tra khả năng từ chối đúng ca ngoài phạm vi / thiếu bằng chứng.

Kết quả chi tiết được hiển thị trực tiếp ngay bên dưới các cell và đồng thời xuất ra **`evaluation/results/ragas_evaluation_report.md`**.

In [1]:
# 1. Khởi tạo đường dẫn dự án và nạp biến môi trường (.env)
from __future__ import annotations
import os
import sys
from pathlib import Path
from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'evaluation':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

load_dotenv(PROJECT_ROOT / '.env', override=True)
print('Project root:', PROJECT_ROOT)
print('GOOGLE_API_KEY configured:', bool(os.getenv('GOOGLE_API_KEY')))
print('NVIDIA_API_KEY configured:', bool(os.getenv('NVIDIA_API_KEY')))
print('QDRANT_URL:', os.getenv('QDRANT_URL', 'http://127.0.0.1:6333'))

Project root: C:\Study\Acne Advisor AI\acne-agent-system
GOOGLE_API_KEY configured: True
NVIDIA_API_KEY configured: True
QDRANT_URL: http://localhost:6333


In [2]:
# 2. Cấu hình tham số đánh giá và kiểm tra Knowledge Build hiện hành
import json
import pandas as pd
from evaluation.ragas_runner import (
    DEFAULT_DATASET_PATH,
    DEFAULT_EMBEDDING_MODEL,
    DEFAULT_FALLBACK_JUDGE_MODELS,
    DEFAULT_JUDGE_MODEL,
    DEFAULT_JUDGE_PROVIDER,
    DEFAULT_RESULTS_DIR,
    export_evaluation_artifacts,
    load_evaluation_dataset,
    render_markdown_report,
    run_system_on_dataset,
    score_records_with_ragas,
    summarize_evaluation,
)
from src.knowledge.versioning import resolve_active_knowledge_build_id

DATASET_PATH = PROJECT_ROOT / DEFAULT_DATASET_PATH
RESULTS_DIR = PROJECT_ROOT / DEFAULT_RESULTS_DIR
SYSTEM_MODEL = os.getenv('GOOGLE_MODEL', 'gemini-3.5-flash-lite')  # Model trả lời của hệ thống
JUDGE_PROVIDER = os.getenv('RAGAS_JUDGE_PROVIDER', DEFAULT_JUDGE_PROVIDER)
JUDGE_MODEL = os.getenv('RAGAS_JUDGE_MODEL', DEFAULT_JUDGE_MODEL)
JUDGE_LABEL = JUDGE_MODEL if '/' in JUDGE_MODEL else f'{JUDGE_PROVIDER}/{JUDGE_MODEL}'
_raw_fallbacks = os.getenv('RAGAS_FALLBACK_JUDGE_MODELS', '')
FALLBACK_JUDGE_MODELS = (
    tuple(m.strip() for m in _raw_fallbacks.split(',') if m.strip())
    if _raw_fallbacks.strip()
    else DEFAULT_FALLBACK_JUDGE_MODELS
)
EMBEDDING_MODEL = DEFAULT_EMBEDDING_MODEL
REQUEST_DELAY_SECONDS = 1.0
MAX_CASES = None

kb_build_id = resolve_active_knowledge_build_id()
print('Active KB Build ID:', kb_build_id)
print('System Model:', SYSTEM_MODEL, '| Judge:', JUDGE_LABEL, '| Fallback Judges:', FALLBACK_JUDGE_MODELS)

C:\Study\Acne Advisor AI\acne-agent-system\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Active KB Build ID: b768fae999391f47760a
System Model: gemini-3.5-flash-lite | Judge: nvidia/nemotron-3-ultra-550b-a55b | Fallback Judges: ('mistralai/mistral-large-4-1t-a52b',)


In [3]:
# 3. Nạp và xem tổng quan bộ 50 câu hỏi đánh giá
from IPython.display import display

dataset_meta, cases = load_evaluation_dataset(DATASET_PATH)
if MAX_CASES is not None:
    cases = cases[:MAX_CASES]

df_cases = pd.DataFrame([
    {'case_id': c.case_id, 'category': c.category, 'expected': c.expected_behavior, 'question': c.user_input}
    for c in cases
])
print(f'Tổng số câu hỏi sẽ chạy: {len(cases)}')
display(df_cases)

Tổng số câu hỏi sẽ chạy: 50


,case_id,category,expected,question
0,NEW-SYM-01,symptoms_and_types,answer,Mụn nang (cyst) và mụn cục (nodule) có đặc điể...
1,NEW-SYM-02,symptoms_and_types,answer,Trứng cá mạch lươn thường gặp ở đối tượng nào ...
2,NEW-SYM-03,symptoms_and_types,answer,Sẹo lõm sau mụn trứng cá được chia thành những...
3,NEW-SYM-04,symptoms_and_types,answer,Trứng cá tối cấp (Acne fulminans) có biểu hiện...
4,NEW-SYM-05,symptoms_and_types,answer,Mụn trứng cá mức độ nhẹ và mức độ trung bình –...
5,NEW-SYM-06,symptoms_and_types,answer,Trứng cá do cào gãi có biểu hiện gì và thường ...
6,NEW-SYM-07,symptoms_and_types,answer,Mụn mủ đầu mặt ở trẻ sơ sinh (neonatal cephali...
7,NEW-SYM-08,symptoms_and_types,answer,Mụn trứng cá ở người trưởng thành (adult acne)...
8,NEW-CAU-01,causes_and_mechanisms,answer,Khí hậu nóng ẩm và thời tiết mùa hè ảnh hưởng ...
9,NEW-CAU-02,causes_and_mechanisms,answer,Chu trình chăm sóc da không phù hợp có thể tác...


In [4]:
# 4. Chạy hệ thống Acne Advisor AI trên bộ câu hỏi (tắt cache để đo thực tế)
from IPython.display import display

system_records = await run_system_on_dataset(
    cases,
    llm_provider='gemini',
    llm_model=SYSTEM_MODEL,
    verbose=True,
)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
(RESULTS_DIR / 'ragas_system_records.json').write_text(
    json.dumps(system_records, ensure_ascii=False, indent=2),
    encoding='utf-8',
)
df_sys = pd.DataFrame([
    {
        'case_id': r['case_id'],
        'action': r['action'],
        'contexts': len(r['retrieved_contexts']),
        'sources': len(r['sources']),
        'latency_ms': r['latency_ms'],
        'error': r['error'],
    }
    for r in system_records
])
display(df_sys)


Loading weights: 100%|██████████| 393/393 [00:03<00:00, 103.60it/s]


[System 1/50] NEW-SYM-01 | action=generate | contexts=1 | 18535ms
[System 2/50] NEW-SYM-02 | action=generate | contexts=1 | 11449ms
[System 3/50] NEW-SYM-03 | action=generate | contexts=1 | 11461ms
[System 4/50] NEW-SYM-04 | action=generate | contexts=2 | 12190ms
[System 5/50] NEW-SYM-05 | action=generate | contexts=1 | 11650ms
[System 6/50] NEW-SYM-06 | action=generate | contexts=1 | 11767ms
[System 7/50] NEW-SYM-07 | action=generate | contexts=2 | 11477ms
[System 8/50] NEW-SYM-08 | action=generate | contexts=3 | 11623ms
[System 9/50] NEW-CAU-01 | action=generate | contexts=2 | 10989ms
[System 10/50] NEW-CAU-02 | action=generate | contexts=2 | 11871ms
[System 11/50] NEW-CAU-03 | action=generate | contexts=2 | 12047ms
[System 12/50] NEW-CAU-04 | action=generate | contexts=2 | 11541ms
[System 13/50] NEW-CAU-05 | action=generate | contexts=1 | 16616ms
[System 14/50] NEW-CAU-06 | action=generate | contexts=1 | 12668ms
[System 15/50] NEW-CAU-07 | action=generate | contexts=1 | 11635ms
[Sys

,case_id,action,contexts,sources,latency_ms,error
0,NEW-SYM-01,generate,1,1,18534.53,None
1,NEW-SYM-02,generate,1,1,11448.97,None
2,NEW-SYM-03,generate,1,1,11461.13,None
3,NEW-SYM-04,generate,2,2,12189.62,None
4,NEW-SYM-05,generate,1,2,11650.09,None
5,NEW-SYM-06,generate,1,1,11767.11,None
6,NEW-SYM-07,generate,2,1,11477.16,None
7,NEW-SYM-08,generate,3,4,11622.59,None
8,NEW-CAU-01,generate,2,1,10988.70,None
9,NEW-CAU-02,generate,2,1,11871.25,None


In [5]:
# 5. Chấm điểm bằng RAGAS (5 chỉ số cốt lõi + Safe Abstention Accuracy)
from IPython.display import display

scored_records = await score_records_with_ragas(
    system_records,
    judge_model=JUDGE_MODEL,
    judge_provider=JUDGE_PROVIDER,
    fallback_judge_models=FALLBACK_JUDGE_MODELS,
    embedding_model=EMBEDDING_MODEL,
    request_delay_seconds=REQUEST_DELAY_SECONDS,
    verbose=True,
)
summary = summarize_evaluation(
    scored_records,
    dataset_meta=dataset_meta,
    judge_model=JUDGE_LABEL,
    embedding_model=EMBEDDING_MODEL,
    kb_build_id=kb_build_id,
)
df_summary = pd.DataFrame([
    {'metric': k, **v} for k, v in summary['ragas_summary'].items()
])
display(df_summary)
df_per_case = pd.DataFrame([
    {
        'case_id': r['case_id'],
        'category': r['category'],
        'action': r['action'],
        **(r.get('ragas_scores') or {}),
        'abstention_ok': r.get('abstention_correct'),
        'latency_ms': r['latency_ms'],
    }
    for r in scored_records
])
display(df_per_case)
print('Safe Abstention:', summary['safe_abstention'])
print('Latency (ms):', summary['latency_ms'])
print('Token Usage:', summary['token_usage'])

[RAGAS 1/50] NEW-SYM-01 | faith=1.000, rel=0.979, prec=1.000, rec=0.667, corr=0.826
[RAGAS 2/50] NEW-SYM-02 | faith=1.000, rel=0.980, prec=1.000, rec=1.000, corr=0.922
[RAGAS 3/50] NEW-SYM-03 | faith=1.000, rel=0.974, prec=1.000, rec=1.000, corr=0.986
[RAGAS 4/50] NEW-SYM-04 | faith=1.000, rel=0.927, prec=1.000, rec=1.000, corr=0.723
[RAGAS 5/50] NEW-SYM-05 | faith=1.000, rel=0.929, prec=1.000, rec=1.000, corr=0.906
[RAGAS 6/50] NEW-SYM-06 | faith=1.000, rel=0.945, prec=1.000, rec=1.000, corr=0.981
[RAGAS 7/50] NEW-SYM-07 | faith=1.000, rel=0.941, prec=1.000, rec=1.000, corr=0.862
[RAGAS 8/50] NEW-SYM-08 | faith=1.000, rel=0.932, prec=0.833, rec=1.000, corr=0.784
[RAGAS 9/50] NEW-CAU-01 | faith=1.000, rel=0.882, prec=1.000, rec=1.000, corr=0.733
[RAGAS 10/50] NEW-CAU-02 | faith=0.500, rel=0.852, prec=1.000, rec=1.000, corr=0.663
[RAGAS 11/50] NEW-CAU-03 | faith=1.000, rel=1.000, prec=1.000, rec=1.000, corr=0.837
[RAGAS 12/50] NEW-CAU-04 | faith=1.000, rel=0.966, prec=1.000, rec=1.000, 

,metric,mean,min,max,count
0,faithfulness,0.9097,0.2000,1.0000,41
1,answer_relevancy,0.9311,0.8251,1.0000,42
2,context_precision,0.9802,0.5000,1.0000,42
3,context_recall,0.8938,0.0000,1.0000,42
4,answer_correctness,0.7389,0.3526,0.9931,42


,case_id,category,action,faithfulness,answer_relevancy,context_precision,context_recall,answer_correctness,abstention_ok,latency_ms
0,NEW-SYM-01,symptoms_and_types,generate,1.0000,0.9787,1.0000,0.6667,0.8259,None,18534.53
1,NEW-SYM-02,symptoms_and_types,generate,1.0000,0.9803,1.0000,1.0000,0.9224,None,11448.97
2,NEW-SYM-03,symptoms_and_types,generate,1.0000,0.9741,1.0000,1.0000,0.9864,None,11461.13
3,NEW-SYM-04,symptoms_and_types,generate,1.0000,0.9273,1.0000,1.0000,0.7227,None,12189.62
4,NEW-SYM-05,symptoms_and_types,generate,1.0000,0.9293,1.0000,1.0000,0.9062,None,11650.09
5,NEW-SYM-06,symptoms_and_types,generate,1.0000,0.9454,1.0000,1.0000,0.9808,None,11767.11
6,NEW-SYM-07,symptoms_and_types,generate,1.0000,0.9410,1.0000,1.0000,0.8615,None,11477.16
7,NEW-SYM-08,symptoms_and_types,generate,1.0000,0.9316,0.8333,1.0000,0.7838,None,11622.59
8,NEW-CAU-01,causes_and_mechanisms,generate,1.0000,0.8816,1.0000,1.0000,0.7326,None,10988.70
9,NEW-CAU-02,causes_and_mechanisms,generate,0.5000,0.8521,1.0000,1.0000,0.6633,None,11871.25


Safe Abstention: {'correct': 8, 'total': 8, 'accuracy': 1.0}
Latency (ms): {'mean': 10597.8, 'p50': 11505.0, 'p95': 16615.9}
Token Usage: {'input': 99987, 'output': 6682, 'total': 106669}


In [6]:
# 6. Xuất báo cáo chi tiết ra file Markdown (.md), JSON thô và hiển thị trực tiếp tại Notebook
from IPython.display import Markdown, display

report_path, raw_path = export_evaluation_artifacts(
    summary,
    output_dir=RESULTS_DIR,
    report_filename='ragas_evaluation_report.md',
    raw_filename='ragas_evaluation_raw.json',
)
print('Đã xuất báo cáo Markdown:', report_path)
print('Đã xuất dữ liệu thô JSON:', raw_path)
report_md = render_markdown_report(summary)
preview_md = report_md.split('## 5. Chi tiết câu trả lời')[0].strip()
display(Markdown(preview_md))

Đã xuất báo cáo Markdown: C:\Study\Acne Advisor AI\acne-agent-system\evaluation\results\ragas_evaluation_report.md
Đã xuất dữ liệu thô JSON: C:\Study\Acne Advisor AI\acne-agent-system\evaluation\results\ragas_evaluation_raw.json


# Báo cáo Đánh giá Hệ thống Acne Advisor AI (RAGAS)

- **Thời điểm chạy (UTC):** `2026-11-25T04:29:00.204788+00:00`
- **Bộ dữ liệu đánh giá:** `acne_advisor_ragas_benchmark` (50 câu hỏi: 42 câu trả lời, 8 câu từ chối an toàn)
- **Judge LLM (RAGAS):** `nvidia/nemotron-3-ultra-550b-a55b`
- **Embedding Model:** `models/gemini-embedding-2`
- **Knowledge Build ID:** `b768fae999391f47760a`

## 1. Tổng quan chỉ số RAGAS

Phương pháp đánh giá tham chiếu bài báo *RAGAS: Automated Evaluation of Retrieval Augmented Generation* (Es et al., 2023, `arXiv:2309.15217`). Thang điểm từ `0.000` đến `1.000` (càng cao càng tốt).

| Chỉ số | Trung bình | Thấp nhất | Cao nhất | Số ca chấm | Đánh giá |
| --- | --- | --- | --- | --- | --- |
| Độ trung thực với bằng chứng (Faithfulness) | 0.910 | 0.200 | 1.000 | 41 | Tốt (>= 0.80) |
| Độ liên quan câu trả lời (Answer Relevancy) | 0.931 | 0.825 | 1.000 | 42 | Tốt (>= 0.80) |
| Độ chính xác ngữ cảnh (Context Precision) | 0.980 | 0.500 | 1.000 | 42 | Tốt (>= 0.80) |
| Độ bao phủ ngữ cảnh (Context Recall) | 0.894 | 0.000 | 1.000 | 42 | Tốt (>= 0.80) |
| Độ chính xác câu trả lời (Answer Correctness) | 0.739 | 0.353 | 0.993 | 42 | Khá (0.65 - 0.79) |

## 2. Chỉ số Vận hành & An toàn

| Hạng mục | Kết quả |
| --- | --- |
| Tỷ lệ từ chối an toàn (Safe Abstention Accuracy) | 8/8 (1.000) |
| Độ trễ trung bình (Mean Latency) | 10597.8 ms |
| Độ trễ trung vị (P50 Latency) | 11505.0 ms |
| Độ trễ P95 (P95 Latency) | 16615.9 ms |
| Tổng token tiêu thụ (Generation) | input=99987, output=6682, total=106669 |
| Số lỗi runtime | 0 |

## 3. Phân tích theo nhóm câu hỏi

| Nhóm câu hỏi | Số ca | Faithfulness | Answer Relevancy | Context Precision | Context Recall | Answer Correctness | Độ trễ TB (ms) |
| --- | --- | --- | --- | --- | --- | --- | --- |
| `symptoms_and_types` | 8 | 1.000 | 0.951 | 0.979 | 0.958 | 0.874 | 12518.9 |
| `causes_and_mechanisms` | 7 | 0.929 | 0.938 | 1.000 | 0.786 | 0.735 | 12481.1 |
| `comparison` | 6 | 0.952 | 0.907 | 1.000 | 0.967 | 0.640 | 11836.9 |
| `treatment_and_actives` | 9 | 0.936 | 0.940 | 1.000 | 0.952 | 0.811 | 11961.0 |
| `safety_and_precautions` | 7 | 0.804 | 0.918 | 0.929 | 0.929 | 0.692 | 11308.1 |
| `lifestyle_and_care` | 5 | 0.758 | 0.920 | 0.967 | 0.700 | 0.584 | 12060.3 |
| `safe_abstention` | 8 | N/A | N/A | N/A | N/A | N/A | 3030.4 |

## 4. Bảng điểm chi tiết từng câu hỏi

| Case ID | Nhóm | Hành động | Faithfulness | Relevancy | Precision | Recall | Correctness | Từ chối đúng | Độ trễ (ms) |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| `NEW-SYM-01` | `symptoms_and_types` | `generate` | 1.000 | 0.979 | 1.000 | 0.667 | 0.826 | - | 18534.53 |
| `NEW-SYM-02` | `symptoms_and_types` | `generate` | 1.000 | 0.980 | 1.000 | 1.000 | 0.922 | - | 11448.97 |
| `NEW-SYM-03` | `symptoms_and_types` | `generate` | 1.000 | 0.974 | 1.000 | 1.000 | 0.986 | - | 11461.13 |
| `NEW-SYM-04` | `symptoms_and_types` | `generate` | 1.000 | 0.927 | 1.000 | 1.000 | 0.723 | - | 12189.62 |
| `NEW-SYM-05` | `symptoms_and_types` | `generate` | 1.000 | 0.929 | 1.000 | 1.000 | 0.906 | - | 11650.09 |
| `NEW-SYM-06` | `symptoms_and_types` | `generate` | 1.000 | 0.945 | 1.000 | 1.000 | 0.981 | - | 11767.11 |
| `NEW-SYM-07` | `symptoms_and_types` | `generate` | 1.000 | 0.941 | 1.000 | 1.000 | 0.862 | - | 11477.16 |
| `NEW-SYM-08` | `symptoms_and_types` | `generate` | 1.000 | 0.932 | 0.833 | 1.000 | 0.784 | - | 11622.59 |
| `NEW-CAU-01` | `causes_and_mechanisms` | `generate` | 1.000 | 0.882 | 1.000 | 1.000 | 0.733 | - | 10988.7 |
| `NEW-CAU-02` | `causes_and_mechanisms` | `generate` | 0.500 | 0.852 | 1.000 | 1.000 | 0.663 | - | 11871.25 |
| `NEW-CAU-03` | `causes_and_mechanisms` | `generate` | 1.000 | 1.000 | 1.000 | 1.000 | 0.837 | - | 12047.35 |
| `NEW-CAU-04` | `causes_and_mechanisms` | `generate` | 1.000 | 0.966 | 1.000 | 1.000 | 0.733 | - | 11541.28 |
| `NEW-CAU-05` | `causes_and_mechanisms` | `generate` | 1.000 | 0.950 | 1.000 | 0.500 | 0.836 | - | 16615.93 |
| `NEW-CAU-06` | `causes_and_mechanisms` | `generate` | 1.000 | 0.935 | 1.000 | 0.000 | 0.353 | - | 12668.45 |
| `NEW-CAU-07` | `causes_and_mechanisms` | `generate` | 1.000 | 0.982 | 1.000 | 1.000 | 0.987 | - | 11634.83 |
| `NEW-CMP-01` | `comparison` | `generate` | 1.000 | 0.904 | 1.000 | 1.000 | 0.487 | - | 12531.44 |
| `NEW-CMP-02` | `comparison` | `generate` | 0.714 | 0.940 | 1.000 | 1.000 | 0.625 | - | 11845.36 |
| `NEW-CMP-03` | `comparison` | `generate` | 1.000 | 0.941 | 1.000 | 1.000 | 0.680 | - | 11658.58 |
| `NEW-CMP-04` | `comparison` | `generate` | 1.000 | 0.889 | 1.000 | 1.000 | 0.747 | - | 11532.8 |
| `NEW-CMP-05` | `comparison` | `generate` | 1.000 | 0.871 | 1.000 | 0.800 | 0.465 | - | 12393.71 |
| `NEW-CMP-06` | `comparison` | `generate` | 1.000 | 0.896 | 1.000 | 1.000 | 0.838 | - | 11059.56 |
| `NEW-TRT-01` | `treatment_and_actives` | `generate` | 1.000 | 0.989 | 1.000 | 1.000 | 0.993 | - | 11022.76 |
| `NEW-TRT-02` | `treatment_and_actives` | `generate` | 1.000 | 0.861 | 1.000 | 1.000 | 0.993 | - | 11777.21 |
| `NEW-TRT-03` | `treatment_and_actives` | `generate` | 0.900 | 0.957 | 1.000 | 1.000 | 0.724 | - | 11909.37 |
| `NEW-TRT-04` | `treatment_and_actives` | `generate` | 1.000 | 0.961 | 1.000 | 0.571 | 0.826 | - | 11157.41 |
| `NEW-TRT-05` | `treatment_and_actives` | `generate` | 1.000 | 0.938 | 1.000 | 1.000 | 0.861 | - | 9910.1 |
| `NEW-TRT-06` | `treatment_and_actives` | `generate` | 1.000 | 0.951 | 1.000 | 1.000 | 0.587 | - | 17375.43 |
| `NEW-TRT-07` | `treatment_and_actives` | `generate` | 0.857 | 0.825 | 1.000 | 1.000 | 0.847 | - | 11904.66 |
| `NEW-TRT-08` | `treatment_and_actives` | `generate` | 0.667 | 0.983 | 1.000 | 1.000 | 0.739 | - | 11147.7 |
| `NEW-TRT-09` | `treatment_and_actives` | `generate` | 1.000 | 1.000 | 1.000 | 1.000 | 0.727 | - | 11444.15 |
| `NEW-SAF-01` | `safety_and_precautions` | `generate` | 0.500 | 0.919 | 1.000 | 1.000 | 0.565 | - | 11269.72 |
| `NEW-SAF-02` | `safety_and_precautions` | `generate` | 0.571 | 0.846 | 1.000 | 1.000 | 0.927 | - | 11175.2 |
| `NEW-SAF-03` | `safety_and_precautions` | `generate` | 1.000 | 0.984 | 1.000 | 1.000 | 0.442 | - | 11192.6 |
| `NEW-SAF-04` | `safety_and_precautions` | `generate` | 1.000 | 0.923 | 1.000 | 1.000 | 0.598 | - | 12225.08 |
| `NEW-SAF-05` | `safety_and_precautions` | `generate` | 0.833 | 0.926 | 1.000 | 0.500 | 0.635 | - | 10990.91 |
| `NEW-SAF-06` | `safety_and_precautions` | `generate` | 0.923 | 0.907 | 1.000 | 1.000 | 0.947 | - | 11094.66 |
| `NEW-SAF-07` | `safety_and_precautions` | `generate` | 0.800 | 0.919 | 0.500 | 1.000 | 0.730 | - | 11208.62 |
| `NEW-CAR-01` | `lifestyle_and_care` | `generate` | 1.000 | 0.858 | 1.000 | 1.000 | 0.685 | - | 11274.8 |
| `NEW-CAR-02` | `lifestyle_and_care` | `generate` | 0.833 | 0.958 | 1.000 | 0.500 | 0.503 | - | 11906.57 |
| `NEW-CAR-03` | `lifestyle_and_care` | `generate` | 0.200 | 0.904 | 1.000 | 0.000 | 0.602 | - | 11567.19 |
| `NEW-CAR-04` | `lifestyle_and_care` | `generate` | 1.000 | 0.955 | 0.833 | 1.000 | 0.550 | - | 13287.86 |
| `NEW-CAR-05` | `lifestyle_and_care` | `generate` | N/A | 0.927 | 1.000 | 1.000 | 0.579 | - | 12265.17 |
| `NEW-ABS-01` | `safe_abstention` | `abstain` | N/A | N/A | N/A | N/A | N/A | Đạt | 6175.35 |
| `NEW-ABS-02` | `safe_abstention` | `abstain` | N/A | N/A | N/A | N/A | N/A | Đạt | 1871.92 |
| `NEW-ABS-03` | `safe_abstention` | `abstain` | N/A | N/A | N/A | N/A | N/A | Đạt | 2080.66 |
| `NEW-ABS-04` | `safe_abstention` | `abstain` | N/A | N/A | N/A | N/A | N/A | Đạt | 2328.87 |
| `NEW-ABS-05` | `safe_abstention` | `abstain` | N/A | N/A | N/A | N/A | N/A | Đạt | 2108.69 |
| `NEW-ABS-06` | `safe_abstention` | `generate` | N/A | N/A | N/A | N/A | N/A | Đạt | 17.37 |
| `NEW-ABS-07` | `safe_abstention` | `abstain` | N/A | N/A | N/A | N/A | N/A | Đạt | 2202.4 |
| `NEW-ABS-08` | `safe_abstention` | `abstain` | N/A | N/A | N/A | N/A | N/A | Đạt | 7457.76 |